In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import torch

REPO_URL = "https://github.com/ituvtu/reid_investigation.git"
REPO_DIR = Path("/kaggle/working/reid_investigation")


def _run_step(step_label: str, command: list[str], allow_failure: bool = False) -> subprocess.CompletedProcess[str]:
    print(f"[{step_label}] START: {' '.join(command)}")
    result = subprocess.run(command, capture_output=True, text=True)

    if result.stdout.strip():
        print(f"[{step_label}] STDOUT:\n{result.stdout.strip()}")
    if result.stderr.strip():
        print(f"[{step_label}] STDERR:\n{result.stderr.strip()}")

    if result.returncode != 0 and not allow_failure:
        raise RuntimeError(f"[{step_label}] FAILED with code {result.returncode};")

    if result.returncode == 0:
        print(f"[{step_label}] DONE;")
    else:
        print(f"[{step_label}] FAILED but continuing due to allow_failure=True;")

    return result


def _write_requirements_without_dinov2(source_path: Path, output_path: Path) -> None:
    lines = source_path.read_text(encoding="utf-8").splitlines()
    filtered = [line for line in lines if "dinov2" not in line.lower()]
    output_path.write_text("\n".join(filtered) + "\n", encoding="utf-8")


print("[1/9] Bootstrap started;")
print(f"Current working directory: {Path.cwd()};")
print(f"Repository target path: {REPO_DIR};")

print("[2/9] Repository clone check;")
if not REPO_DIR.exists():
    _run_step("2/9", ["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    print("[2/9] Repository already exists - clone skipped;")

print("[3/9] Change directory;")
os.chdir(REPO_DIR)
print(f"[3/9] Current working directory: {Path.cwd()};")

print("[4/9] Install uv package manager;")
_run_step("4/9", [sys.executable, "-m", "pip", "install", "-q", "uv"])

print("[5/9] Resolve uv command;")
uv_cli = shutil.which("uv")
if uv_cli is None:
    uv_command = [sys.executable, "-m", "uv"]
    print("[5/9] Using fallback: python -m uv;")
else:
    uv_command = [uv_cli]
    print(f"[5/9] Using uv executable: {uv_cli};")

print("[6/9] Install Python dependencies from requirements.txt;")
requirements_path = Path("requirements.txt")
step6_result = _run_step(
    "6/9",
    uv_command + ["pip", "install", "-r", str(requirements_path), "--system"],
    allow_failure=True,
)

if step6_result.returncode != 0:
    stderr_text = (step6_result.stderr or "").lower()
    is_dinov2_solver_conflict = "dinov2" in stderr_text and "unsatisfiable" in stderr_text

    if is_dinov2_solver_conflict:
        print("[6/9] Detected dinov2 dependency conflict with torchvision - applying fallback;")
        fallback_requirements_path = Path("/tmp/requirements_stage2_no_dinov2.txt")
        _write_requirements_without_dinov2(requirements_path, fallback_requirements_path)

        _run_step(
            "6/9A",
            uv_command + ["pip", "install", "-r", str(fallback_requirements_path), "--system"],
        )

        install_dinov2_optional = os.environ.get("INSTALL_DINOV2_OPTIONAL", "0") == "1"
        if install_dinov2_optional:
            _run_step(
                "6/9B",
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--no-deps",
                    "git+https://github.com/facebookresearch/dinov2.git",
                ],
            )
        else:
            print("[6/9B] Optional dinov2 install skipped; set INSTALL_DINOV2_OPTIONAL=1 to enable;")
    else:
        raise RuntimeError(f"[6/9] FAILED with code {step6_result.returncode};")

print("[7/9] Inject repository path into sys.path;")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[7/9] Path injected: {str(REPO_DIR) in sys.path};")

print("[8/9] Detect SoccerNet dataset paths;")

def _normalize_soccernet_root(path: Path) -> Path:
    normalized = path
    if path.name.lower() in {"train", "valid", "test"}:
        normalized = path.parent
    return normalized


input_root = Path("/kaggle/input")
preferred_train_path = Path("/kaggle/input/datasets/theoviel/soccernet-tracking/train")
soccernet_dirs: list[Path] = []

if preferred_train_path.exists():
    print(f"[8/9] Preferred path found: {preferred_train_path};")
    soccernet_dirs.append(preferred_train_path)
else:
    print(f"[8/9] Preferred path not found: {preferred_train_path};")

if input_root.exists():
    print(f"[8/9] Scanning: {input_root};")
    soccernet_dirs.extend(
        path for path in input_root.rglob("*")
        if path.is_dir() and "soccernet" in path.name.lower()
    )
else:
    print(f"[8/9] Input root not found: {input_root};")

unique_dirs: list[Path] = []
seen: set[str] = set()
for path in soccernet_dirs:
    key = str(path.resolve())
    if key in seen:
        continue
    unique_dirs.append(path)
    seen.add(key)

if unique_dirs and not os.environ.get("SOCCERNET_ROOT_DIR"):
    os.environ["SOCCERNET_ROOT_DIR"] = str(_normalize_soccernet_root(unique_dirs[0]))

print(f"[8/9] SoccerNet candidates found: {len(unique_dirs)};")
if unique_dirs:
    for path in unique_dirs[:10]:
        print(f"- {path};")

print("[9/9] GPU and environment summary;")
gpu_detected = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_detected else "No GPU detected"

print(f"GPU detected: {gpu_detected};")
print(f"GPU name: {gpu_name};")
print(f"SOCCERNET_ROOT_DIR: {os.environ.get('SOCCERNET_ROOT_DIR', '<not-set>')};")
print("[9/9] Bootstrap completed;")

In [ ]:
from pathlib import Path
import os

DEFAULT_REPO_URL = "https://github.com/ituvtu/reid_investigation.git"
os.environ.setdefault("REID_REPO_URL", DEFAULT_REPO_URL)

# Optional overrides:
# os.environ["GITHUB_TOKEN"] = "<your_pat>"
# os.environ["SOCCERNET_ROOT_DIR"] = "/content/drive/MyDrive/SoccerNet"
# os.environ["SOCCERNET_PASSWORD"] = "<password_if_required>"

print("Working directory:", Path.cwd())
print("models exists:", os.path.exists("models"))
print("configs exists:", os.path.exists("configs"))
print("REID_REPO_URL:", os.environ.get("REID_REPO_URL", ""))

In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys


def _mask_text(text: str, secrets: list[str] | None = None) -> str:
    if not secrets:
        return text
    masked = text
    for secret in secrets:
        if secret:
            masked = masked.replace(secret, "***")
    return masked


def _run(cmd, secrets: list[str] | None = None):
    printable_cmd = _mask_text(" ".join(cmd), secrets)
    print("$", printable_cmd)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(_mask_text(result.stdout, secrets))
    if result.returncode != 0:
        stderr_text = result.stderr.strip() or "(no stderr)"
        stderr_text = _mask_text(stderr_text, secrets)
        raise RuntimeError(f"Command failed ({result.returncode}): {printable_cmd}\n{stderr_text}")


def _is_project_root(path: Path) -> bool:
    return (path / "configs").exists() and (path / "models").exists()


def _find_project_root(start_path: Path) -> Path | None:
    candidates = [start_path, *start_path.parents]
    for candidate in candidates:
        if _is_project_root(candidate):
            return candidate
    return None


def _scan_content_for_project_root(content_dir: Path) -> Path | None:
    # Detect a workspace synced by Colab extension under an unknown directory name.
    for candidate in content_dir.rglob("*"):
        if candidate.is_dir() and _is_project_root(candidate):
            return candidate
    return None


def _build_clone_url(repo_url: str) -> tuple[str, list[str]]:
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token and repo_url.startswith("https://github.com/"):
        # Personal access token flow for private repositories in Colab.
        authenticated_url = repo_url.replace("https://", f"https://{token}@", 1)
        return authenticated_url, [token]
    return repo_url, []


def _resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    local_root = _find_project_root(cwd)
    if local_root is not None:
        return local_root

    content_dir = Path("/content")
    if content_dir.exists():
        known_candidates = [
            content_dir / "reid_investigation",
            content_dir / "drive" / "MyDrive" / "reid_investigation",
        ]
        for candidate in known_candidates:
            if _is_project_root(candidate):
                return candidate

        scanned_root = _scan_content_for_project_root(content_dir)
        if scanned_root is not None:
            return scanned_root

        default_repo_url = "https://github.com/ituvtu/reid_investigation.git"
        repo_url = os.environ.get("REID_REPO_URL", default_repo_url).strip()
        clone_url, clone_secrets = _build_clone_url(repo_url)
        clone_target = content_dir / "reid_investigation"
        if not clone_target.exists():
            print(f"Cloning repository for Colab: {repo_url};")
            try:
                _run(["git", "clone", clone_url, str(clone_target)], secrets=clone_secrets)
            except Exception as error:
                raise RuntimeError(
                    "Unable to clone repository to /content/reid_investigation; "
                    "for a private repository set %env GITHUB_TOKEN=<token> or use /content/drive/MyDrive/reid_investigation; "
                    f"details: {error}"
                ) from error

            if _is_project_root(clone_target):
                return clone_target

        if _is_project_root(clone_target):
            return clone_target

        raise RuntimeError(
            "Project source was not found in Colab (/content/reid_investigation or /content/drive/MyDrive/reid_investigation). "
            "If the repository is private, set %env GITHUB_TOKEN=<token> and optionally %env REID_REPO_URL=<url> before running Cell 2;"
        )

    raise RuntimeError(
        "Could not find a project root with configs and models directories; check the working directory;"
    )


PROJECT_ROOT = _resolve_project_root()
get_ipython().run_line_magic("cd", str(PROJECT_ROOT))

module_to_package = {
    "numpy": "numpy",
    "torch": "torch",
    "ultralytics": "ultralytics",
    "motmetrics": "motmetrics",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "yaml": "PyYAML",
    "cv2": "opencv-python",
    "torchreid": "torchreid",
    "gdown": "gdown",
    "SoccerNet": "SoccerNet",
}
required_modules = tuple(module_to_package.keys())
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]

print(f"Python executable: {sys.executable};")
print(f"Python version: {sys.version.split()[0]};")
print(f"Project root: {PROJECT_ROOT};")


def _detect_uv_cli() -> list[str] | None:
    uv_path = shutil.which("uv")
    if uv_path:
        return [uv_path]

    try:
        _run([sys.executable, "-m", "uv", "--version"])
        return [sys.executable, "-m", "uv"]
    except Exception:
        pass

    try:
        _run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
        _run([sys.executable, "-m", "pip", "install", "uv"])
        _run([sys.executable, "-m", "uv", "--version"])
        return [sys.executable, "-m", "uv"]
    except Exception:
        return None


def _install_packages(packages: list[str]) -> None:
    uv_cli = _detect_uv_cli()
    if uv_cli is not None:
        print("Installing dependencies via uv;")
        _run(uv_cli + ["pip", "install", "--python", sys.executable, *packages])
        return

    print("uv is unavailable - fallback to pip;")
    _run([sys.executable, "-m", "pip", "install", *packages])


if missing_modules:
    packages_to_install = [module_to_package[name] for name in missing_modules]
    print(f"Installing dependencies for active kernel: {', '.join(packages_to_install)};")
    _install_packages(packages_to_install)

failed_imports = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
    except Exception:
        failed_imports.append(module_name)

if failed_imports:
    raise RuntimeError(
        "Failed to import modules after installation: "
        + ", ".join(failed_imports)
        + "; restart the kernel and run Cell 2 again;"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
if os.path.exists("/content") and "/content" not in sys.path:
    sys.path.append("/content")

print(f"Import path attached: {str(PROJECT_ROOT) in sys.path};")
print("Dependency status: ready;")

In [ ]:
from pathlib import Path

from models.detectors import YOLODetector
from models.reid import OSNetReID
from models.trackers import ByteTrackTracker
from utils import load_stage2_reid_config, stage2_component_mappings

config = load_stage2_reid_config("configs/stage2_reid.yaml")
detector_config, tracker_config, reid_config = stage2_component_mappings(config)

requested_weights = str(detector_config.get("weights_path", "")).strip()
requested_model = str(detector_config.get("model_name", "")).strip()
weights_path = Path(requested_weights) if requested_weights else None

if requested_model.lower().startswith("yolo26"):
    detector_config["weights_path"] = requested_model
    print(f"YOLOv26 library mode enabled: source={requested_model};")
elif requested_weights and weights_path is not None and not weights_path.exists():
    raise FileNotFoundError(
        f"Weights file not found: {requested_weights}; provide a valid path or use a model name source."
    )

detector = YOLODetector.from_config(detector_config)
detector.load()
detector.warmup(image_size=(640, 640), runs=1)

shared_device = detector.device
tracker_config["device"] = shared_device
reid_config["device"] = shared_device

tracker = ByteTrackTracker.from_config(tracker_config)
tracker.initialize()

reid_model = OSNetReID.from_config(reid_config)
reid_model.load()

print(f"Detector device: {detector.device}")
print(f"Tracker device: {tracker.device}")
print(f"ReID device: {reid_model.device}")
print(f"Experiment: {config.experiment_name}")

if not str(detector.device).startswith("cuda"):
    print("Warning: CUDA was not found - running on CPU;")

In [ ]:
import csv
import os
from pathlib import Path
import subprocess

import cv2
from core.base_tracker import Track
from utils import SoccerNetLoader

# Mount Drive before any dataset IO so /content/drive/MyDrive is truly persistent.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception as error:
    print(f'Drive mount step skipped or failed: {error}')

if config.dataset is None or config.dataset.soccernet is None:
    raise ValueError("Missing dataset.soccernet configuration in configs/stage2_reid.yaml")

soccernet_cfg = config.dataset.soccernet
override_root = os.environ.get("SOCCERNET_ROOT_DIR", "").strip()
override_password = os.environ.get("SOCCERNET_PASSWORD", "").strip() or soccernet_cfg.password

candidate_roots = [
    Path(override_root).expanduser() if override_root else None,
    Path(soccernet_cfg.root_dir).expanduser(),
    Path('/content/drive/MyDrive/SoccerNet'),
    Path('/content/SoccerNet'),
]

resolved_root: Path | None = None
for candidate in candidate_roots:
    if candidate is not None and candidate.exists():
        resolved_root = candidate
        break

target_root = resolved_root or (Path(override_root).expanduser() if override_root else Path(soccernet_cfg.root_dir).expanduser())
dataset_mapping = {
    'root_dir': str(target_root),
    'subset': soccernet_cfg.subset,
    'split': list(soccernet_cfg.split),
    'password': override_password,
    'auto_download': True,
}

soccernet_loader = SoccerNetLoader.from_config(dataset_mapping)


def _extract_zip_archives(root: Path) -> int:
    zip_files = sorted(root.rglob('*.zip')) if root.exists() else []
    if not zip_files:
        return 0

    print(f'Found {len(zip_files)} zip archives; extracting in place;')
    extracted_count = 0
    for zip_path in zip_files:
        cmd = f"unzip -n '{zip_path.as_posix()}' -d '{zip_path.parent.as_posix()}'"
        print(f'$ {cmd}')
        result = subprocess.run(['bash', '-lc', cmd], check=False)
        if result.returncode == 0:
            extracted_count += 1
    return extracted_count


def _find_video_files(root: Path) -> list[Path]:
    extensions = ('.mp4', '.avi', '.mov', '.mkv')
    return sorted([path for path in root.rglob('*') if path.is_file() and path.suffix.lower() in extensions])


def _find_annotation_files(root: Path) -> tuple[list[Path], list[Path], list[Path]]:
    json_files = sorted(root.rglob('*.json'))
    csv_files = sorted(root.rglob('*.csv'))
    txt_files = sorted(root.rglob('*.txt'))
    return json_files, csv_files, txt_files


def _load_mot_table_tracks(path: Path) -> dict[int, list[Track]]:
    tracks_by_frame: dict[int, list[Track]] = {}
    with path.open('r', encoding='utf-8', newline='') as file:
        reader = csv.reader(file)
        for row in reader:
            if len(row) < 6:
                continue
            try:
                frame_id = int(float(row[0]))
                track_id = int(float(row[1]))
                x = float(row[2])
                y = float(row[3])
                w = float(row[4])
                h = float(row[5])
                confidence = float(row[6]) if len(row) > 6 and row[6] != '' else 1.0
            except Exception:
                continue

            tracks_by_frame.setdefault(frame_id, []).append(
                Track(
                    track_id=track_id,
                    bbox_xyxy=(x, y, x + w, y + h),
                    confidence=confidence,
                    class_id=0,
                    embedding=None,
                )
            )
    return tracks_by_frame


def iter_image_sequence_frames(image_files: list[Path], max_frames: int | None = None, stride: int = 1):
    produced = 0
    for image_path in image_files:
        frame_id = None
        try:
            frame_id = int(image_path.stem)
        except Exception:
            continue

        if frame_id % stride != 0:
            continue

        frame = cv2.imread(str(image_path))
        if frame is None:
            continue

        yield frame_id, frame
        produced += 1

        if max_frames is not None and produced >= max_frames:
            break


def _choose_sequence_annotation(sequence_root: Path) -> Path | None:
    gt_dir = sequence_root / 'gt'
    candidates: list[Path] = []

    if gt_dir.exists():
        candidates.extend(sorted(gt_dir.glob('*.json')))
        candidates.extend(sorted(gt_dir.glob('*.csv')))
        candidates.extend(sorted(gt_dir.glob('*.txt')))

    candidates.extend(sorted(sequence_root.glob('*.json')))
    candidates.extend(sorted(sequence_root.glob('*.csv')))
    candidates.extend(sorted(sequence_root.glob('*.txt')))

    filtered_candidates = [
        path for path in candidates
        if path.is_file() and ('gt' in path.name.lower() or 'tracking' in path.name.lower() or path.suffix.lower() == '.json')
    ]

    if filtered_candidates:
        return filtered_candidates[0]
    return candidates[0] if candidates else None


def _load_annotation_tracks(annotation_path: Path) -> dict[int, list[Track]]:
    if annotation_path.suffix.lower() == '.json':
        annotation_payload = soccernet_loader.load_tracking_annotations(annotation_path=annotation_path)
        return soccernet_loader.map_tracking_annotations_to_tracks(annotation_payload)

    if annotation_path.suffix.lower() in {'.csv', '.txt'}:
        return _load_mot_table_tracks(annotation_path)

    raise RuntimeError(f'Unsupported annotation format: {annotation_path.suffix}')


def _discover_sequence_sources(root: Path) -> list[dict]:
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    img_dirs = sorted([path for path in root.rglob('img1') if path.is_dir()])
    discovered_sources: list[dict] = []

    for img_dir in img_dirs:
        sequence_root = img_dir.parent
        image_files = sorted(
            [path for path in img_dir.iterdir() if path.is_file() and path.suffix.lower() in image_extensions],
            key=lambda path: path.name,
        )
        if not image_files:
            continue

        annotation_path = _choose_sequence_annotation(sequence_root)
        if annotation_path is None:
            continue

        gt_tracks = _load_annotation_tracks(annotation_path)
        if not gt_tracks:
            continue

        try:
            source_name = sequence_root.relative_to(root).as_posix()
        except Exception:
            source_name = sequence_root.name

        discovered_sources.append(
            {
                'source_name': source_name,
                'frame_source_kind': 'image_sequence',
                'source_path': img_dir,
                'image_files': image_files,
                'annotation_path': annotation_path,
                'gt_tracks_by_frame': gt_tracks,
            }
        )

    return discovered_sources


try:
    dataset_root = soccernet_loader.ensure_dataset()
except Exception as error:
    print(f'Loader fallback activated: {error};')
    _extract_zip_archives(target_root)
    dataset_root = target_root

video_files = _find_video_files(target_root)
json_files, csv_files, txt_files = _find_annotation_files(target_root)

print(f'SoccerNet root used: {dataset_mapping["root_dir"]}')
print(f'SoccerNet directory: {dataset_root}')
print(f'Videos found: {len(video_files)}')
print(f'JSON annotations found: {len(json_files)}')
print(f'CSV annotations found: {len(csv_files)}')
print(f'TXT annotations found: {len(txt_files)}')

requested_source_count = int(os.environ.get('STAGE2_SOURCE_COUNT', '10'))
if requested_source_count < 1:
    raise ValueError('STAGE2_SOURCE_COUNT must be >= 1;')

available_sequence_sources = _discover_sequence_sources(target_root)
if not available_sequence_sources:
    raise RuntimeError('No valid tracking sequences with img1 and annotations were found;')

evaluation_sources = available_sequence_sources[:requested_source_count]
if len(evaluation_sources) < requested_source_count:
    raise RuntimeError(
        f'Requested {requested_source_count} sources but only found {len(evaluation_sources)} valid sequences;'
    )

# Compatibility variables for downstream cells and ad-hoc checks.
primary_source = evaluation_sources[0]
frame_source_kind = primary_source['frame_source_kind']
sequence_image_files = primary_source['image_files']
video_path = primary_source['source_path']
annotation_path = primary_source['annotation_path']
gt_tracks_by_frame = primary_source['gt_tracks_by_frame']

print(f'Selected sources for evaluation: {len(evaluation_sources)}')
for index, source in enumerate(evaluation_sources, start=1):
    gt_frames = len(source['gt_tracks_by_frame'])
    image_count = len(source['image_files'])
    print(f"{index}. {source['source_name']} | images={image_count} | gt_frames={gt_frames} | ann={source['annotation_path']}")

In [ ]:
from pathlib import Path

# 1) Ensure Drive is mounted.
try:
    from google.colab import drive, auth  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception as error:
    print(f'Colab modules unavailable: {error}')

# 2) Show current Google account used by the notebook.
try:
    import google.auth
    from googleapiclient.discovery import build  # type: ignore

    auth.authenticate_user()
    creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
    service = build('drive', 'v3', credentials=creds)

    about = service.about().get(fields='user(displayName,emailAddress)').execute()
    user_info = about.get('user', {})
    print('Active Google account:')
    print('Name:', user_info.get('displayName', '(unknown)'))
    print('Email:', user_info.get('emailAddress', '(unknown)'))
except Exception as error:
    print(f'Could not read account info via Drive API: {error}')

# 3) Show local mounted path status.
dataset_root = Path('/content/drive/MyDrive/SoccerNet')
print('Mounted dataset path exists:', dataset_root.exists())
print('Mounted dataset path:', dataset_root)

# 4) Find SoccerNet folders in Google Drive and print direct links.
try:
    query = "name='SoccerNet' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    response = service.files().list(
        q=query,
        fields='files(id,name,parents)',
        pageSize=20,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    folders = response.get('files', [])

    if not folders:
        print('No folder named SoccerNet found in this Google account.')
    else:
        print('Drive folder links for SoccerNet:')
        for folder in folders:
            folder_id = folder.get('id', '')
            folder_name = folder.get('name', 'SoccerNet')
            print(f"- {folder_name}: https://drive.google.com/drive/folders/{folder_id}")
except Exception as error:
    print(f'Could not list SoccerNet folders: {error}')

print('Tip: if email is not yours, run drive.mount with force_remount=True and re-authenticate.')

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import subprocess

drive_root = Path('/content/drive/MyDrive/SoccerNet')
tracking_root = drive_root / 'tracking'

print('Drive mounted:', Path('/content/drive/MyDrive').exists())
print('SoccerNet path exists:', drive_root.exists())
print('Tracking path exists:', tracking_root.exists())

print('\nTop-level SoccerNet listing:')
subprocess.run(['bash', '-lc', "ls -lah /content/drive/MyDrive/SoccerNet 2>/dev/null || true"], check=False)

print('\nTracking listing:')
subprocess.run(['bash', '-lc', "ls -lah /content/drive/MyDrive/SoccerNet/tracking 2>/dev/null || true"], check=False)

print('\nZip sizes:')
subprocess.run(['bash', '-lc', "du -sh /content/drive/MyDrive/SoccerNet/tracking/*.zip 2>/dev/null || true"], check=False)

marker = drive_root / f'COLAB_MARKER_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.txt'
marker.parent.mkdir(parents=True, exist_ok=True)
marker.write_text('created_from_colab=' + datetime.utcnow().isoformat() + 'Z\n', encoding='utf-8')
print('\nCreated marker:', marker)
print('Now search this marker filename in drive.google.com;')

In [ ]:
from pathlib import Path
import os
import subprocess

# Ensure Google Drive is mounted in Colab.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception:
    print('google.colab is unavailable; skip mount step;')

# Copy from temporary location only when it exists.
source_root = Path(os.environ.get('SOCCERNET_SOURCE_ROOT', '/content/SoccerNet')).expanduser()
drive_root = Path('/content/drive/MyDrive/SoccerNet').expanduser()

drive_root.parent.mkdir(parents=True, exist_ok=True)
drive_root.mkdir(parents=True, exist_ok=True)

# Detect if data already exists on Drive (files or archives).
drive_has_payload = any(drive_root.rglob('*.mp4')) or any(drive_root.rglob('*.json')) or any(drive_root.rglob('*.zip'))

if drive_has_payload:
    print(f'Drive already has SoccerNet payload at: {drive_root}; skip copy;')
elif str(source_root).startswith('/content/drive/MyDrive'):
    print(f'Source already points to Drive: {source_root};')
elif source_root.exists():
    rsync_cmd = (
        f"rsync -ah --info=progress2 --partial --append-verify "
        f"'{source_root.as_posix()}/' '{drive_root.as_posix()}/'"
    )
    print('Copying dataset to Google Drive with resume support;')
    print(f'$ {rsync_cmd}')
    subprocess.run(['bash', '-lc', rsync_cmd], check=True)
else:
    print(
        f"Source dataset not found at {source_root}; nothing to copy. "
        "If Cell 4 downloaded to Drive directly, this is expected;"
    )

print('Storage check;')
subprocess.run(['bash', '-lc', f"du -sh '{source_root.as_posix()}' 2>/dev/null || true"], check=False)
subprocess.run(['bash', '-lc', f"du -sh '{drive_root.as_posix()}' 2>/dev/null || true"], check=False)

# Point next cells to persistent location.
os.environ['SOCCERNET_ROOT_DIR'] = drive_root.as_posix()
print(f'SOCCERNET_ROOT_DIR set to: {os.environ["SOCCERNET_ROOT_DIR"]};')

In [ ]:
import importlib
import utils.metrics as metrics_module

importlib.reload(metrics_module)
from utils import compute_mot_id_metrics

print('Reloaded utils.metrics with NumPy 2 compatibility shim;')

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from core.base_tracker import Track
from utils import compute_mot_id_metrics

# Add compatibility for NumPy 2.x;
if not hasattr(np, 'asfarray'):
    def _np_asfarray_compat(values, dtype=float):
        return np.asarray(values, dtype=dtype)

    np.asfarray = _np_asfarray_compat

try:
    import cv2
except Exception:
    cv2 = None
    print('Warning: OpenCV is unavailable - visualization will be limited;')

if 'evaluation_sources' not in globals() or not evaluation_sources:
    raise RuntimeError('evaluation_sources is missing - run Cell 4 first;')

if 'reid_model' not in globals():
    raise RuntimeError('reid_model is missing - run Cell 3 first;')

if 'tracker' not in globals():
    raise RuntimeError('tracker is missing - run Cell 3 first;')

max_frames_per_source = int(os.environ.get('STAGE2_MAX_FRAMES', '120'))
frame_stride = int(os.environ.get('STAGE2_FRAME_STRIDE', '2'))
iou_threshold = float(os.environ.get('STAGE2_IOU_THRESHOLD', '0.5'))

if max_frames_per_source < 1:
    raise ValueError('STAGE2_MAX_FRAMES must be >= 1;')
if frame_stride < 1:
    raise ValueError('STAGE2_FRAME_STRIDE must be >= 1;')

association_alpha_value = float(getattr(tracker, 'association_alpha', tracker_config.get('association_alpha', 0.5)))
default_input_size = reid_config.get('input_size', [256, 128])
reid_input_height = int(getattr(reid_model, 'input_height', int(default_input_size[0])))
reid_input_width = int(getattr(reid_model, 'input_width', int(default_input_size[1])))

print(f'Current association alpha parameter: {association_alpha_value:.3f};')
print(f'ReID crop size: {reid_input_height}x{reid_input_width};')


def _offset_tracks_by_frame_and_id(
    tracks_by_frame: dict[int, list[Track]],
    frame_offset: int,
    track_offset: int,
) -> dict[int, list[Track]]:
    remapped: dict[int, list[Track]] = {}
    for frame_id, tracks in tracks_by_frame.items():
        remapped_frame = int(frame_id) + frame_offset
        remapped[remapped_frame] = [
            Track(
                track_id=int(track.track_id) + track_offset,
                bbox_xyxy=track.bbox_xyxy,
                confidence=float(track.confidence),
                class_id=int(track.class_id),
                embedding=track.embedding,
            )
            for track in tracks
        ]
    return remapped


def _extract_crops_for_reid(frame: np.ndarray, detections) -> list[np.ndarray]:
    # Clip bounding boxes to frame boundaries and resize crops to the OSNet input size;
    if frame is None or frame.size == 0:
        return [np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8) for _ in detections]

    frame_height, frame_width = frame.shape[:2]
    if frame_height < 1 or frame_width < 1:
        return [np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8) for _ in detections]

    crops: list[np.ndarray] = []
    for detection in detections:
        x1, y1, x2, y2 = detection.bbox_xyxy

        left = int(np.clip(np.floor(x1), 0, frame_width - 1))
        top = int(np.clip(np.floor(y1), 0, frame_height - 1))
        right = int(np.clip(np.ceil(x2), left + 1, frame_width))
        bottom = int(np.clip(np.ceil(y2), top + 1, frame_height))

        crop = frame[top:bottom, left:right]
        if cv2 is None or crop.size == 0:
            resized_crop = np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8)
        else:
            resized_crop = cv2.resize(
                crop,
                (reid_input_width, reid_input_height),
                interpolation=cv2.INTER_LINEAR,
            )

        crops.append(resized_crop)

    return crops


preview_frames: list[tuple[str, int, np.ndarray]] = []
per_source_metrics_state: list[dict] = []
combined_ground_truth: dict[int, list[Track]] = {}
combined_predictions: dict[int, list[Track]] = {}

for source_index, source in enumerate(evaluation_sources):
    source_name = str(source['source_name'])
    source_kind = str(source['frame_source_kind'])
    source_gt_tracks_by_frame = source['gt_tracks_by_frame']

    if source_kind == 'video':
        frame_iterator = soccernet_loader.iter_video_frames(
            video_path=source['source_path'],
            max_frames=max_frames_per_source,
            stride=frame_stride,
        )
    else:
        frame_iterator = iter_image_sequence_frames(
            image_files=source['image_files'],
            max_frames=max_frames_per_source,
            stride=frame_stride,
        )

    tracker.initialize()

    source_latencies: list[float] = []
    source_predictions_by_frame: dict[int, list[Track]] = {}

    for frame_index, frame in frame_iterator:
        frame_start = time.perf_counter()
        detections = detector.predict(frame)

        if detections:
            crops = _extract_crops_for_reid(frame, detections)
            embeddings = reid_model.extract(crops)
        else:
            embeddings = None

        tracks = tracker.update(detections=detections, embeddings=embeddings, frame_index=frame_index)
        source_latencies.append(time.perf_counter() - frame_start)
        source_predictions_by_frame[frame_index] = tracks

        if len(preview_frames) < 3:
            vis_frame = frame.copy()
            if cv2 is not None:
                cv2.putText(
                    vis_frame,
                    f'alpha={association_alpha_value:.2f}',
                    (12, 24),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA,
                )
                for track in tracks:
                    x1, y1, x2, y2 = [int(value) for value in track.bbox_xyxy]
                    cv2.rectangle(vis_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(
                        vis_frame,
                        f'ID {track.track_id}',
                        (x1, max(44, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA,
                    )
            preview_frames.append((source_name, int(frame_index), vis_frame))

    if not source_predictions_by_frame:
        raise RuntimeError(f'No frames were processed for source: {source_name};')

    source_metric_values = compute_mot_id_metrics(
        ground_truth_by_frame=source_gt_tracks_by_frame,
        predictions_by_frame=source_predictions_by_frame,
        iou_threshold=iou_threshold,
    )

    source_fps = (1.0 / float(np.mean(source_latencies))) if source_latencies else 0.0
    processed_frames = len(source_predictions_by_frame)

    per_source_metrics_state.append(
        {
            'Source': source_name,
            'MOTA': float(source_metric_values['MOTA']),
            'IDF1': float(source_metric_values['IDF1']),
            'ID Swaps': int(source_metric_values['ID Swaps']),
            'FPS': float(source_fps),
            'Processed Frames': int(processed_frames),
            'Association Alpha': float(association_alpha_value),
        }
    )

    # Isolate ID spaces across sources;
    frame_offset = (source_index + 1) * 1_000_000
    track_offset = (source_index + 1) * 100_000

    combined_ground_truth.update(
        _offset_tracks_by_frame_and_id(source_gt_tracks_by_frame, frame_offset=frame_offset, track_offset=track_offset)
    )
    combined_predictions.update(
        _offset_tracks_by_frame_and_id(source_predictions_by_frame, frame_offset=frame_offset, track_offset=track_offset)
    )

if len(per_source_metrics_state) != len(evaluation_sources):
    raise RuntimeError('Not all selected sources produced valid metrics;')

aggregate_metric_values = compute_mot_id_metrics(
    ground_truth_by_frame=combined_ground_truth,
    predictions_by_frame=combined_predictions,
    iou_threshold=iou_threshold,
)

average_fps = float(np.mean([item['FPS'] for item in per_source_metrics_state])) if per_source_metrics_state else 0.0

metrics_state = {
    'MOTA': float(aggregate_metric_values['MOTA']),
    'IDF1': float(aggregate_metric_values['IDF1']),
    'ID Swaps': int(aggregate_metric_values['ID Swaps']),
    'FPS': float(average_fps),
    'Sources Evaluated': int(len(per_source_metrics_state)),
    'Association Alpha': float(association_alpha_value),
}

preview_count = len(preview_frames)
if preview_count > 0:
    fig, axes = plt.subplots(1, preview_count, figsize=(5 * preview_count, 4))
    if preview_count == 1:
        axes = [axes]

    for index in range(preview_count):
        source_name, frame_index, frame_to_show = preview_frames[index]
        if cv2 is not None:
            frame_to_show = cv2.cvtColor(frame_to_show, cv2.COLOR_BGR2RGB)
        axes[index].imshow(frame_to_show)
        axes[index].set_title(f'{source_name} - frame {frame_index} - alpha {association_alpha_value:.2f}')
        axes[index].axis('off')

    plt.tight_layout()

print('Status: per-source and aggregate metric computation completed;')
print(f"Sources evaluated: {metrics_state['Sources Evaluated']};")

In [ ]:
from pathlib import Path

import pandas as pd

if 'metrics_state' not in globals():
    raise RuntimeError('metrics_state is missing - run Cell 9 first;')
if 'per_source_metrics_state' not in globals():
    raise RuntimeError('per_source_metrics_state is missing - run Cell 9 first;')

aggregate_rows = [
    {'Metric': 'MOTA', 'Value': float(metrics_state['MOTA'])},
    {'Metric': 'IDF1', 'Value': float(metrics_state['IDF1'])},
    {'Metric': 'ID Swaps', 'Value': int(metrics_state['ID Swaps'])},
    {'Metric': 'FPS (avg across sources)', 'Value': float(metrics_state['FPS'])},
    {'Metric': 'Sources Evaluated', 'Value': int(metrics_state['Sources Evaluated'])},
    {'Metric': 'Association Alpha', 'Value': float(metrics_state['Association Alpha'])},
]

aggregate_metrics_table = pd.DataFrame(aggregate_rows)
per_source_metrics_table = pd.DataFrame(per_source_metrics_state)

aggregate_display = aggregate_metrics_table.copy()
aggregate_display['Value'] = aggregate_display['Value'].apply(
    lambda value: f'{value:.3f}' if isinstance(value, float) else value
)

per_source_display = per_source_metrics_table.copy()
for column in ['MOTA', 'IDF1', 'FPS', 'Association Alpha']:
    if column in per_source_display.columns:
        per_source_display[column] = per_source_display[column].map(lambda value: f'{float(value):.3f}')

print('Metrics: aggregate summary across selected sources;')
display(aggregate_display)

print('Metrics: per-source breakdown;')
display(per_source_display)

outputs_dir = Path('outputs')
outputs_dir.mkdir(parents=True, exist_ok=True)

aggregate_metrics_path = outputs_dir / 'stage2_metrics.csv'
per_source_metrics_path = outputs_dir / 'stage2_metrics_by_source.csv'

aggregate_metrics_table.to_csv(aggregate_metrics_path, index=False)
per_source_metrics_table.to_csv(per_source_metrics_path, index=False)

print(f'Saved aggregate metrics CSV: {aggregate_metrics_path};')
print(f'Saved per-source metrics CSV: {per_source_metrics_path};')